In [ ]:
# 1. Clone repository
!rm -rf /kaggle/working/Project-CLARITY
!git clone -b Dileep https://github.com/Git-Dileep/Project-CLARITY.git /kaggle/working/Project-CLARITY

# 2. Enter workstream directory
%cd /kaggle/working/Project-CLARITY/detection/workstream

# 3. Install required packages
!pip install -r requirements.txt


In [ ]:
import os, sys, json, time, shutil, glob, torch, joblib
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

# Set directory paths
base_dir = Path("/kaggle/working/Project-CLARITY/detection/workstream")
os.chdir(base_dir)
if str(base_dir) not in sys.path:
    sys.path.insert(0, str(base_dir))

# 1. Output directories
reports_dir = Path("/kaggle/working/outputs/reports")
models_dir = Path("/kaggle/working/outputs/models")
reports_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

# 2. Target the exact ACI-IoT-2023.csv flow dataset
dataset_path = "/kaggle/input/datasets/emilynack/aci-iot-network-traffic-dataset-2023/ACI-IoT-2023.csv"
if not os.path.exists(dataset_path):
    candidates = glob.glob("/kaggle/input/**/ACI-IoT-2023.csv", recursive=True)
    if not candidates:
        raise FileNotFoundError("Could not find ACI-IoT-2023.csv in /kaggle/input. Please ensure the dataset is attached.")
    dataset_path = candidates[0]

print(f"\n--> Using correct flow dataset at: {dataset_path}")

# 3. Write Kaggle configuration
kaggle_config = {
  "dataset_name": "ACI-IoT-Network-2023",
  "is_synthetic": False,
  "input_path": dataset_path,
  "target_column": "Label",
  "flow_id_column": "Flow ID",
  "timestamp_column": "Timestamp",
  "device_id_column": None,
  "metadata_columns": ["Source IP", "Destination IP", "Flow ID", "Timestamp", "source_file", "sample_id"],
  "drop_columns": ["Source IP", "Destination IP", "Flow ID", "Timestamp"],
  "numeric_columns": None,
  "categorical_columns": None,
  "max_rows_per_class": 50000,
  "test_size": 0.2,
  "random_seed": 42,
  "chunksize": 250000,
  "traditional": {
    "rf_n_estimators": 100,
    "rf_max_depth": 15,
    "xgb_n_estimators": 100,
    "xgb_max_depth": 6,
    "xgb_learning_rate": 0.1,
    "mlp_hidden_layer_sizes": [64, 32],
    "mlp_max_iter": 50,
    "mlp_early_stopping": True
  },
  "dl": {
    "enabled": True,
    "batch_size": 256,
    "epochs": 10,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "cnn_channels": 32,
    "lstm_hidden": 32,
    "transformer_dim": 32,
    "transformer_heads": 4,
    "early_stopping_patience": 3,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
  },
  "explainability": {
    "top_n_features": 10,
    "background_samples": 50,
    "traditional_explainer": "TreeExplainer",
    "dl_explainer": "GradientExplainer"
  },
  "pilot": {
    "pilot_size": 20,
    "min_classes": 5,
    "stratified": True
  },
  "reports_dir": "/kaggle/working/outputs/reports",
  "models_dir": "/kaggle/working/outputs/models",
  "data_dir": str(Path(dataset_path).parent)
}

with open("config/kaggle_aci_2023.json", "w") as f:
    json.dump(kaggle_config, f, indent=2)

# 4. Verified imports
from src.config import WorkstreamConfig
from src.audit_datasets import run_audit
from src.data_loader import load_dataset
from src.preprocessing import prepare_data_and_splits
from src.train_traditional import train_traditional_models
from src.train_dl import train_dl_models
from src.export_pilot import export_pilot_alerts

cfg = WorkstreamConfig.load("config/kaggle_aci_2023.json")

print("\n" + "="*60)
print("[STAGE 1/5] Running Dataset Audit...")
print("="*60)
run_audit(cfg)

print("\n" + "="*60)
print("[STAGE 2/5] Loading & Preprocessing ACI-IoT-2023...")
print("="*60)
df, _ = load_dataset(cfg)

# In-memory sanitization: convert inf, -inf, and extreme floats to NaN
df = df.replace(["Infinity", "-Infinity", "inf", "-inf", np.inf, -np.inf], np.nan)
for col in df.select_dtypes(include=[np.number]).columns:
    df.loc[df[col] > 1e300, col] = np.nan
    df.loc[df[col] < -1e300, col] = np.nan

splits = prepare_data_and_splits(df, cfg)

print("\n" + "="*60)
print("[STAGE 3/5] Training Traditional Ensemble (RF, XGB, MLP)...")
print("="*60)
trad_res = train_traditional_models(splits, cfg)

print("\n" + "="*60)
print("[STAGE 4/5] Training Deep Learning Ensemble (CNN, Bi-LSTM, Transformer)...")
print("="*60)
dl_res = train_dl_models(splits, cfg)

print("\n" + "="*60)
print("[STAGE 5/5] Generating Matched Pilot Alerts (Consensus SHAP) & Dual Strategy Exports...")
print("="*60)
dl_res["models"]["Transformer"].to("cpu")
export_res = export_pilot_alerts(cfg, base_dir=base_dir, trad_result=trad_res, dl_result=dl_res, splits=splits)

print("\n" + "="*60)
print("CREATING ZIP ARCHIVE FOR DOWNLOAD...")
print("="*60)
shutil.make_archive("/kaggle/working/clarity_real_aci_results", "zip", "/kaggle/working/outputs")

print("\n🎉 SUCCESS! PIPELINE COMPLETED IN FULL.")
print(f"Traditional Alerts: {export_res.get('traditional_alerts_count')} (Valid: {export_res.get('traditional_valid')})")
print(f"DL Alerts: {export_res.get('dl_alerts_count')} (Valid: {export_res.get('dl_valid')})")
print("Zip archive created at: /kaggle/working/clarity_real_aci_results.zip")
